In [6]:
import os
import pandas as pd
import joblib
import re
from scipy import sparse


class PricePredictor:   
    def _init_(self):
        self.model = joblib.load("final_model.pkl")
        self.tfidf = joblib.load("tfidf.pkl")
        
    def clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]','',text)
        text = re.sub(r'\s+',' ',text).strip()
        return text

    def predict(self, sample_id, catalog_content, image_link):

        cleaned_content = self.clean_text(catalog_content)
        word_count=len(str(cleaned_content).split())
        char_count=len(str(cleaned_content))
        digit_count=sum(c.isdigit() for c in str(cleaned_content))
        
        x_text = self.tfidf.transform([cleaned_content])
        
        extra_features_list = [[word_count, char_count, digit_count]]
        x_extra = sparse.csr_matrix(extra_features_list)
        X_combined = sparse.hstack([x_text,x_extra])

        y_pred = self.model.predict(X_combined)

        return y_pred[0]
    



DATASET_FOLDER=r"G:\My Drive\Classroom\68e8d1d70b66d_student_resource\student_resource\dataset"

# Read test data
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))

price_predictor = PricePredictor()



# Apply predictor function to each row
test['price'] = test.apply(
    lambda row: price_predictor.predict(row['sample_id'], row['catalog_content'], row['image_link']), 
    axis=1
)

# Select only required columns for output
output_df = test[['sample_id', 'price']]

# Save predictions
output_filename = os.path.join(DATASET_FOLDER, 'test_out.csv')
output_df.to_csv(output_filename, index=False)

print(f"Predictions saved to {output_filename}")
print(f"Total predictions: {len(output_df)}")
print(f"Sample predictions:\n{output_df.head()}")

AttributeError: 'PricePredictor' object has no attribute 'tfidf'